In [ ]:
import pandas as pd

In [ ]:
weather_df = pd.read_csv('../../../data/raw/weather_mm_sumry2.csv')
# 컬럼 확인
weather_df.columns
weather_df.head()

In [ ]:
# 필요한 컬럼 추출
weather_df = weather_df.filter(items=['year', 'month', 'stn_id', 'stn_ko', 'rn_day'])

# 강수량 결측값 확인 / 해당 월에 강수량이 관측되지 않음 (== 0)
weather_df[weather_df['rn_day'].isnull()]
weather_df['rn_day'] = weather_df['rn_day'].fillna(0)

# 지역 ID 결측값 확인 / 월만 수집 -> 삭제 처리 
weather_df[weather_df['stn_id'].isnull()]
weather_df = weather_df.dropna(how='any')
weather_df.info()

In [ ]:
# 지역코드 부여를 위한 컬럼 값 확인
unique_values = weather_df.loc[:, 'stn_ko'].unique()
print(unique_values)

In [ ]:
address_df = pd.read_csv('../../../data/raw/META_weather_place.csv')

# 결측값 처리
address_df[address_df['지점주소'].isnull()]
address_df['지점주소'] = address_df['지점주소'].fillna('충청북도 청주시 흥덕구')
address_df[address_df['지점주소'].isnull()]

In [ ]:
address_df = address_df.rename(columns = {'지점명': 'stn_ko', '지점주소': 'address'})
address_df = address_df.filter(items=['stn_ko', 'address'])

address_df

In [ ]:
address_all_df = pd.merge(weather_df, address_df, on = 'stn_ko', how = 'left')

In [ ]:
address_all_df['SIDO'] = address_all_df['address'].str.split().str[0]
address_all_df['SIGUNGU'] = address_all_df['address'].str.split().str[1]
address_all_df.loc[address_all_df['address'].str.contains('경상남도 창원시.*성산구', na=False), 'ADD_FNM'] = '경상남도 창원시 성산구'

In [ ]:
address_all_df['ADD_FNM'] = address_all_df['SIDO'] + ' ' + address_all_df['SIGUNGU']
address_all_df.info()

In [ ]:
# 예외 처리
address_all_df.loc[address_all_df['address'].str.contains('경상남도 창원시.*성산구', na=False), 'ADD_FNM'] = '경상남도 창원시 성산구'
address_all_df.loc[address_all_df['address'].str.contains('충청북도 청주시.*흥덕구', na=False), 'ADD_FNM'] = '충청북도 청주시 흥덕구'
address_all_df.loc[address_all_df['address'].str.contains('경기도 수원시.*권선구', na=False), 'ADD_FNM'] = '경기도 수원시 권선구'
address_all_df.loc[address_all_df['address'].str.contains('전북특별자치도 전주시.*덕진구', na=False), 'ADD_FNM'] = '전북특별자치도 전주시 덕진구'
address_all_df.loc[address_all_df['address'].str.contains('전북특별자치도 전주시.*완산구', na=False), 'ADD_FNM'] = '전북특별자치도 전주시 완산구'
address_all_df.loc[address_all_df['address'].str.contains('충청남도 천안시.*동남구', na=False), 'ADD_FNM'] = '충청남도 천안시 동남구' 
address_all_df.loc[address_all_df['address'].str.contains('경상북도 포항시.*남구', na=False), 'ADD_FNM'] = '경상북도 포항시 남구'
address_all_df.loc[address_all_df['address'].str.contains('세종특별자치시.*새롬동', na=False), 'ADD_FNM'] = '세종특별자치시 전체'

address_all_df[address_all_df['ADD_FNM'] == '전북특별자치도 전주시 덕진구']

In [ ]:
sigungu_df = pd.read_csv('data/sigungu_code.csv')
sigungu_df['ADD_FNM'] = sigungu_df['시도명'] + ' ' + sigungu_df['시군구명']
sigungu_df

In [ ]:
sigungu_df['ADD_FNM'] = sigungu_df['ADD_FNM'].str.strip()
address_all_df['ADD_FNM'] = address_all_df['ADD_FNM'].str.strip()
sigungu_df

In [ ]:
final_df = pd.merge(address_all_df, sigungu_df, on = 'ADD_FNM', how = 'left')
final_df = final_df.filter(items=[ '지역코드', 'year', 'month','rn_day'])
final_df = final_df.rename(columns = {'지역코드': 'SIGUNGU', 'rn_day': 'RAIN_TOTAL', 'year': 'FLOOD_YEAR', 'month': 'FLOOD_MONTH'})

final_df

In [ ]:
final_df.info()

In [ ]:
final_df.to_csv('../../../data/processed/weather_rain_data.csv', index=False, encoding='utf-8-sig')